### Сравнение моделей — **`22_history_163_global_features_reconcile`**

На основе [`19_comparison_resist_warnings_extended_per_series.ipynb`](19_comparison_resist_warnings_extended_per_series.ipynb): те же **метрики**, **warnings**, **Excel** (листы `{id} Итог`, `{id} Данные теста`, `{id} По операциям`, `{id} Предупреждения`).

**Данные:** широкий CSV [`tz/history/163_all/history_163_all.csv`](../../tz/history/163_all/history_163_all.csv) → long panel. Исключены месяцы 2023-01…2023-07 и операции без сигнала (см. `EXCLUDED_OPERATION_IDS` + авто-все-нули). **Тест:** июль–декабрь 2025.

**Фичи (train-only статика):** `demand_type` (Syntetos–Boylan), `mean_volume_all_time`, `months_since_first_active` от первого ненулевого в **train**. **Динамика:** календарь, `lag_1..3` (NaN до старта ряда), `rolling_*` только по прошлому, `global_volume_lag_1`. Табличные **global ML** — **рекурсивный** прогноз на 6 шагов (без утечки факта теста в лаги).

**Baseline `fact_predict`:** в CSV нет — колонка **NaN**, метрики baseline в сводке — **NaN**.


In [ ]:
import json
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

_ROOT = Path("..").resolve()
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from pandas.errors import PerformanceWarning

warnings.filterwarnings("ignore", "is_categorical_dtype")
warnings.filterwarnings("ignore", "use_inf_as_na")
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=PerformanceWarning)

HAS_LIGHTGBM = False
try:
    from lightgbm import LGBMRegressor
    HAS_LIGHTGBM = True
except ImportError:
    print("lightgbm не установлен — конфигурации LGBM пропущены")

HAS_XGB = False
try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except ImportError:
    print("xgboost не установлен — конфигурации XGB пропущены")

HAS_CATBOOST = False
try:
    from catboost import CatBoostRegressor
    HAS_CATBOOST = True
except ImportError:
    print("catboost не установлен — CatBoost пропущен")

HAS_STATSFORECAST = False
try:
    import statsforecast  # noqa: F401
    HAS_STATSFORECAST = True
except Exception as e:
    print(f"statsforecast недоступен ({type(e).__name__})")

HAS_HIERARCHICAL = False
try:
    import hierarchicalforecast  # noqa: F401
    HAS_HIERARCHICAL = True
except ImportError:
    print("hierarchicalforecast не установлен — reconcile: только TD-масштабирование")

from statsmodels.tsa.holtwinters import ExponentialSmoothing

%matplotlib inline


In [ ]:
RESIST_IDS = [163]
SUMMARY_TOP_PER_RID = 5
EXCEL_ITOG_TOP = 20

HISTORY_CSV = Path("../tz/history/163_all/history_163_all.csv")
OUT_XLSX = Path("../data/processed/22_history_163_global_features_reconcile.xlsx")

TEST_YEAR = 2025
TEST_MONTHS = [7, 8, 9, 10, 11, 12]
HORIZON = len(TEST_MONTHS)

DROP_DATES_BEFORE = pd.Timestamp("2023-08-01")
EXCLUDED_OPERATION_IDS = {10122, 10123, 10126, 10129, 8641, 8655}

STATSFORECAST_FREQ = "MS"
MONTHLY_SEASON_LENGTH = 12
INTERMITTENT_ZERO_MIN_FRACTION = 0.15

ADI_SMOOTH = 1.32
CV2_ERRATIC = 0.49

FEATURE_SET_NAME = "GLOBAL_V1"
FEATURE_COLS_NUM = [
    "month", "quarter", "days_in_month", "months_since_first_active",
    "lag_1", "lag_2", "lag_3",
    "rolling_mean_3", "rolling_std_3", "rolling_mean_6",
    "global_volume_lag_1", "mean_volume_all_time",
]
CAT_COLS_ML = ["operation_id", "demand_type"]

TABULAR_OP_PLACEHOLDER = "— (все операции)"


In [ ]:
def parse_fact_value(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().replace(" ", "")
    if not s:
        return np.nan
    if "," in s and "." not in s:
        s = s.replace(",", ".")
    elif "," in s and "." in s:
        s = s.replace(".", "").replace(",", ".")
    try:
        return float(s)
    except ValueError:
        return np.nan


def wide_history_to_long(dfw: pd.DataFrame) -> pd.DataFrame:
    pairs = []
    for k in range(1, 300):
        o, f = f"operation_id_{k}", f"fact_value_{k}"
        if o in dfw.columns and f in dfw.columns:
            pairs.append((o, f))
        else:
            break
    if not pairs:
        raise ValueError("Не найдены пары operation_id_k / fact_value_k")
    chunks = []
    for ocol, fcol in pairs:
        sub = dfw[["resist_id", "date", ocol, fcol]].copy()
        sub = sub.rename(columns={ocol: "operation_id", fcol: "fact_value_raw"})
        chunks.append(sub)
    out = pd.concat(chunks, ignore_index=True)
    out["operation_id"] = pd.to_numeric(out["operation_id"], errors="coerce")
    out = out.dropna(subset=["operation_id"])
    out["operation_id"] = out["operation_id"].astype(int)
    out["fact_value"] = out["fact_value_raw"].map(parse_fact_value)
    out = out.drop(columns=["fact_value_raw"])
    out = out.dropna(subset=["fact_value"])
    out["date"] = pd.to_datetime(out["date"], errors="coerce")
    out = out.dropna(subset=["date"])
    out["year"] = out["date"].dt.year.astype(int)
    out["month"] = out["date"].dt.month.astype(int)
    out = out.sort_values(["resist_id", "operation_id", "date"]).reset_index(drop=True)
    return out


df_long = wide_history_to_long(pd.read_csv(HISTORY_CSV, dtype=str))
df_long = df_long[df_long["date"] >= DROP_DATES_BEFORE].copy()
df_long = df_long[~df_long["operation_id"].isin(EXCLUDED_OPERATION_IDS)].copy()

_tot = df_long.groupby("operation_id")["fact_value"].sum()
zero_only = set(_tot[_tot == 0].index.astype(int))
if zero_only:
    print("Дополнительно исключены all-zero операции:", sorted(zero_only)[:25], "..." if len(zero_only) > 25 else "")
    df_long = df_long[~df_long["operation_id"].isin(zero_only)].copy()

df_long["fact_predict"] = np.nan
df_long["date"] = df_long["date"].dt.normalize()
print("Строк long:", len(df_long), "операций:", df_long["operation_id"].nunique())
df_long.head()


In [ ]:
def classify_demand_type(y_train: np.ndarray) -> str:
    y = np.asarray(y_train, dtype=float)
    pos = y[y > 0]
    if len(pos) == 0:
        return "Lumpy"
    mean_nz = float(np.mean(pos))
    cv2 = (float(np.std(pos, ddof=0)) / mean_nz) ** 2 if mean_nz > 0 else 999.0
    idx = np.where(y > 0)[0]
    if len(idx) < 2:
        adi = 999.0
    else:
        gaps = np.diff(np.sort(idx))
        adi = float(np.mean(gaps)) if len(gaps) else 999.0
    if adi < ADI_SMOOTH:
        return "Erratic" if cv2 >= CV2_ERRATIC else "Smooth"
    return "Lumpy" if cv2 >= CV2_ERRATIC else "Smooth"


def train_static_per_operation(df_train: pd.DataFrame):
    mean_vol = df_train.groupby("operation_id")["fact_value"].mean()
    dem = {}
    first_active = {}
    for op, g in df_train.groupby("operation_id"):
        y = g.sort_values("date")["fact_value"].values.astype(float)
        dem[op] = classify_demand_type(y)
        nz = g.loc[g["fact_value"] > 0, "date"]
        first_active[op] = nz.min() if len(nz) else pd.NaT
    return mean_vol, dem, first_active


mask_test = (df_long["year"] == TEST_YEAR) & (df_long["month"].isin(TEST_MONTHS))
mask_train = ~mask_test
df_train_only = df_long.loc[mask_train].copy()

mean_volume_all_time, demand_type_map, first_active_train = train_static_per_operation(df_train_only)
print("Пример demand_type:", list(demand_type_map.items())[:5])


In [ ]:
def month_index_from_dates(dates: pd.Series) -> np.ndarray:
    dt = pd.to_datetime(dates)
    return dt.dt.year.values * 12 + (dt.dt.month.values - 1)


def add_panel_features(
    df: pd.DataFrame,
    mean_vol: pd.Series,
    dem_map: dict,
    first_act: dict,
) -> pd.DataFrame:
    d = df.sort_values(["operation_id", "date"]).copy()
    g = d.groupby("operation_id", group_keys=False)
    fv = g["fact_value"]
    d["lag_1"] = fv.shift(1)
    d["lag_2"] = fv.shift(2)
    d["lag_3"] = fv.shift(3)
    d["rolling_mean_3"] = g["fact_value"].transform(
        lambda s: s.shift(1).rolling(3, min_periods=1).mean()
    )
    d["rolling_std_3"] = g["fact_value"].transform(
        lambda s: s.shift(1).rolling(3, min_periods=1).std()
    )
    d["rolling_mean_6"] = g["fact_value"].transform(
        lambda s: s.shift(1).rolling(6, min_periods=1).mean()
    )
    tot = d.groupby(["resist_id", "date"], as_index=False)["fact_value"].sum()
    tot = tot.sort_values(["resist_id", "date"])
    tot["global_volume_lag_1"] = tot.groupby("resist_id")["fact_value"].shift(1)
    d = d.merge(
        tot[["resist_id", "date", "global_volume_lag_1"]],
        on=["resist_id", "date"],
        how="left",
    )
    d["mean_volume_all_time"] = d["operation_id"].map(mean_vol).fillna(0.0)
    d["demand_type"] = d["operation_id"].map(dem_map).fillna("Smooth")
    mi = month_index_from_dates(d["date"])
    fa = pd.to_datetime(d["operation_id"].map(first_act))
    fa_mi = fa.dt.year * 12 + (fa.dt.month - 1)
    msa = mi - fa_mi.to_numpy(dtype=float)
    d["months_since_first_active"] = np.where(np.isnan(msa), 0, msa)
    d["months_since_first_active"] = d["months_since_first_active"].clip(lower=0)
    d["quarter"] = pd.to_datetime(d["date"]).dt.quarter.astype(int)
    d["days_in_month"] = pd.to_datetime(d["date"]).dt.days_in_month.astype(int)
    return d


df_full = add_panel_features(df_long, mean_volume_all_time, demand_type_map, first_active_train)
df_full.head()


In [ ]:
def calc_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> tuple[float, float, float]:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.maximum(np.asarray(y_pred, dtype=float), 0)
    if len(y_true) == 0:
        return 0.0, 0.0, 0.0
    acc = np.empty_like(y_true)
    mask_zero = y_true == 0
    acc[mask_zero] = np.where(y_pred[mask_zero] == 0, 100.0, 0.0)
    mask_ok = ~mask_zero
    if np.any(mask_ok):
        acc[mask_ok] = np.clip(
            100.0 - np.abs(y_true[mask_ok] - y_pred[mask_ok]) / np.abs(y_true[mask_ok]) * 100.0,
            0, 100,
        )
    tochnost = round(float(np.mean(acc)), 2)
    stab = 0.0
    if tochnost > 0:
        cv = np.std(acc, ddof=0) / tochnost
        stab = round(max(0.0, (1 - cv)) * 100, 2)
    kachestvo = 0.0
    if tochnost + stab > 0:
        kachestvo = round(2 * tochnost * stab / (tochnost + stab), 2)
    return tochnost, stab, kachestvo


def _statsmodels_module_tag(warning) -> str:
    fn = (getattr(warning, "filename", None) or "").lower()
    if not fn:
        return ""
    if "holtwinters" in fn:
        return "holtwinters"
    return ""


def interpret_warning(msg: str, category) -> str:
    cname = getattr(category, "__name__", str(category))
    s = str(msg).lower()
    if "maximum likelihood" in s or "convergence" in cname.lower():
        return "MLE/сходимость: короткая история или плохая спецификация."
    if "runtime" in cname.lower():
        return "RuntimeWarning в вычислениях."
    return "См. сообщение."


def append_warning_rows(rows, *, resist_id, operation_id, model_kind, model_label, phase, n_train_points, caught):
    for w in caught:
        msg = str(w.message)
        cat = w.category
        if issubclass(cat, PerformanceWarning):
            continue
        rows.append(
            {
                "resist_id": resist_id,
                "operation_id": operation_id,
                "model_kind": model_kind,
                "model_label": model_label,
                "phase": phase,
                "warning_category": getattr(cat, "__name__", str(cat)),
                "message": msg,
                "interpretation": interpret_warning(msg, cat),
                "n_train_points": n_train_points if model_kind == "per_series" else "",
                "statsmodels_module": _statsmodels_module_tag(w),
            }
        )


def train_ets_per_series(
    df_full,
    mask_train,
    mask_test,
    resist_id,
    series_config_name,
    warning_rows,
    trend="add",
    damped_trend=False,
    seasonal=None,
    seasonal_periods=None,
):
    test_idx = df_full.loc[mask_test].index
    preds = pd.Series(0.0, index=test_idx)
    for op_id, grp in df_full.groupby("operation_id"):
        train_part = grp[mask_train.reindex(grp.index, fill_value=False)].sort_values("date")
        test_part = grp[mask_test.reindex(grp.index, fill_value=False)].sort_values("date")
        if test_part.empty:
            continue
        y_hist = train_part["fact_value"].values.astype(float)
        n_fc = len(test_part)
        n_tr = len(y_hist)
        try:
            if seasonal is not None and seasonal_periods is not None:
                min_hist = max(4, 2 * int(seasonal_periods))
                if len(y_hist) < min_hist:
                    raise ValueError("too short for seasonal")
            elif len(y_hist) < 4:
                raise ValueError("too short")
            if trend == "mul" and np.any(y_hist <= 0):
                raise ValueError("mul trend requires positive series")
            if seasonal == "mul" and np.any(y_hist <= 0):
                raise ValueError("mul seasonal requires positive series")
            model = ExponentialSmoothing(
                y_hist,
                trend=trend,
                damped_trend=damped_trend,
                seasonal=seasonal,
                seasonal_periods=seasonal_periods if seasonal is not None else None,
            )
            with warnings.catch_warnings(record=True) as w_fit:
                warnings.simplefilter("always")
                fit = model.fit(optimized=True, use_brute=False)
            append_warning_rows(
                warning_rows,
                resist_id=resist_id,
                operation_id=op_id,
                model_kind="per_series",
                model_label=series_config_name,
                phase="fit",
                n_train_points=n_tr,
                caught=w_fit,
            )
            with warnings.catch_warnings(record=True) as w_fc:
                warnings.simplefilter("always")
                fc = fit.forecast(n_fc)
            append_warning_rows(
                warning_rows,
                resist_id=resist_id,
                operation_id=op_id,
                model_kind="per_series",
                model_label=series_config_name,
                phase="forecast",
                n_train_points=n_tr,
                caught=w_fc,
            )
            fc = np.asarray(fc, dtype=float).ravel()
        except Exception:
            fc = np.full(n_fc, np.mean(y_hist) if len(y_hist) > 0 else 0)
        preds.loc[test_part.index] = np.maximum(fc, 0)
    return preds.values


def _sf_train_df(train_part: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "unique_id": "s0",
            "ds": pd.to_datetime(train_part["date"]),
            "y": train_part["fact_value"].values.astype(float),
        }
    )


def _sf_forecast_column(fc_df: pd.DataFrame) -> np.ndarray:
    meta = {"unique_id", "ds"}
    num_cols = [c for c in fc_df.columns if c not in meta]
    if not num_cols:
        raise ValueError("statsforecast: нет колонки прогноза")
    return fc_df[num_cols[0]].values.astype(float)


def _sf_build_model_list(sf_model_name: str, season_len: int):
    from statsforecast.models import CrostonClassic, CrostonSBA, HistoricAverage

    m = sf_model_name
    if m == "CrostonSBA":
        return [CrostonSBA()]
    if m == "CrostonClassic":
        return [CrostonClassic()]
    if m == "HistoricAverage":
        return [HistoricAverage()]
    raise ValueError(f"Неизвестная SF-модель: {sf_model_name}")


def train_statsforecast_intermittent_per_series(
    df_full,
    mask_train,
    mask_test,
    resist_id,
    series_config_name,
    warning_rows,
    *,
    sf_model_name: str,
    season_len: int,
):
    from statsforecast import StatsForecast

    test_idx = df_full.loc[mask_test].index
    preds = pd.Series(0.0, index=test_idx)
    for op_id, grp in df_full.groupby("operation_id"):
        train_part = grp[mask_train.reindex(grp.index, fill_value=False)].sort_values("date")
        test_part = grp[mask_test.reindex(grp.index, fill_value=False)].sort_values("date")
        if test_part.empty:
            continue
        y_hist = train_part["fact_value"].values.astype(float)
        n_fc = len(test_part)
        n_tr = len(y_hist)
        zero_frac = float(np.mean(y_hist == 0)) if n_tr else 0.0
        use_intermittent = zero_frac >= INTERMITTENT_ZERO_MIN_FRACTION
        try:
            if n_tr < 2:
                raise ValueError("too short")
            sf_df = _sf_train_df(train_part)
            if use_intermittent:
                models = _sf_build_model_list(sf_model_name, season_len)
            else:
                models = _sf_build_model_list("HistoricAverage", season_len)
            with warnings.catch_warnings(record=True) as w_fit:
                warnings.simplefilter("always")
                sf = StatsForecast(models=models, freq=STATSFORECAST_FREQ, n_jobs=1)
                sf.fit(sf_df)
            append_warning_rows(
                warning_rows,
                resist_id=resist_id,
                operation_id=op_id,
                model_kind="per_series",
                model_label=series_config_name,
                phase="fit",
                n_train_points=n_tr,
                caught=w_fit,
            )
            with warnings.catch_warnings(record=True) as w_pred:
                warnings.simplefilter("always")
                fc_df = sf.predict(h=n_fc)
            append_warning_rows(
                warning_rows,
                resist_id=resist_id,
                operation_id=op_id,
                model_kind="per_series",
                model_label=series_config_name,
                phase="predict",
                n_train_points=n_tr,
                caught=w_pred,
            )
            fc = _sf_forecast_column(fc_df)
            if fc.shape[0] != n_fc:
                fc = fc[:n_fc] if fc.shape[0] > n_fc else np.resize(fc, n_fc)
        except Exception:
            fc = np.full(n_fc, np.mean(y_hist) if len(y_hist) > 0 else 0)
        preds.loc[test_part.index] = np.maximum(fc, 0)
    return preds.values


In [ ]:
BASE_COLS = [
    "resist_id", "operation_id", "date", "year", "month",
    "fact_value", "fact_predict",
]


def recursive_global_predict(
    model,
    df_base: pd.DataFrame,
    mask_tr: pd.Series,
    mask_te: pd.Series,
    test_dates: list,
    mean_vol,
    dem_map,
    first_act,
    cat_op: list,
    cat_dt: list,
    *,
    fill_nan_for_xgb: bool = False,
    catboost_mode: bool = False,
):
    extended = df_base.loc[mask_tr, BASE_COLS].copy()
    pred_by_idx = {}
    for dt in sorted(test_dates):
        block = df_base.loc[(df_base["date"] == dt) & mask_te, BASE_COLS].copy()
        block["fact_value"] = np.nan
        ops_order = block["operation_id"].values
        tmp = pd.concat([extended, block], ignore_index=True)
        tmp = add_panel_features(tmp, mean_vol, dem_map, first_act)
        fr = tmp.loc[tmp["date"] == dt].drop_duplicates(subset=["operation_id"], keep="last")
        fr = fr.set_index("operation_id").reindex(ops_order).reset_index()
        if catboost_mode and HAS_CATBOOST:
            import catboost as cb
            Xcb = pd.concat(
                [fr[FEATURE_COLS_NUM].fillna(0.0).reset_index(drop=True),
                 fr["operation_id"].astype(str).reset_index(drop=True),
                 fr["demand_type"].astype(str).reset_index(drop=True)],
                axis=1,
            )
            Xcb.columns = FEATURE_COLS_NUM + CAT_COLS_ML
            pool = cb.Pool(data=Xcb, cat_features=CAT_COLS_ML)
            y_hat = model.predict(pool)
        else:
            # LGBM: NaN в числовых лагах сохраняем; XGB (native categorical): только числовые NaN → 0
            X = fr[FEATURE_COLS_NUM + CAT_COLS_ML].copy()
            if fill_nan_for_xgb:
                X[FEATURE_COLS_NUM] = X[FEATURE_COLS_NUM].fillna(0.0)
            X["operation_id"] = pd.Categorical(X["operation_id"].astype(str), categories=cat_op)
            X["demand_type"] = pd.Categorical(X["demand_type"].astype(str), categories=cat_dt)
            y_hat = model.predict(X)
        y_hat = np.maximum(np.asarray(y_hat, dtype=float).ravel(), 0)
        for ix, v in zip(block.index, y_hat):
            pred_by_idx[ix] = v
        block_done = block.copy()
        block_done["fact_value"] = y_hat
        extended = pd.concat([extended, block_done], ignore_index=True)
    ix_te = df_base.loc[mask_te].index
    return np.array([pred_by_idx[i] for i in ix_te], dtype=float)


def forecast_total_top_ets(y_train_total: np.ndarray, h: int) -> np.ndarray:
    y_hist = np.asarray(y_train_total, dtype=float)
    try:
        if len(y_hist) < 4:
            raise ValueError("short")
        m = ExponentialSmoothing(y_hist, trend="add", seasonal=None)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            fit = m.fit(optimized=True, use_brute=False)
        fc = fit.forecast(h)
        return np.maximum(np.asarray(fc, dtype=float).ravel(), 0)
    except Exception:
        mu = np.mean(y_hist) if len(y_hist) else 0.0
        return np.full(h, max(mu, 0.0))


def top_down_reconcile_month(bottom_vec: np.ndarray, y_top: float) -> np.ndarray:
    b = np.maximum(np.asarray(bottom_vec, dtype=float), 0)
    s = b.sum()
    if s <= 0:
        return np.full_like(b, y_top / max(len(b), 1))
    return b * (y_top / s)


SERIES_CONFIGS = {
    "ETS-add": {"fn": "ets", "trend": "add", "damped_trend": False, "seasonal": None, "seasonal_periods": None},
    "ETS-damped": {"fn": "ets", "trend": "add", "damped_trend": True, "seasonal": None, "seasonal_periods": None},
    "ETS-seasonal-add": {
        "fn": "ets",
        "trend": "add",
        "damped_trend": False,
        "seasonal": "add",
        "seasonal_periods": 12,
    },
}
if HAS_STATSFORECAST:
    SERIES_CONFIGS["SF-CrostonSBA"] = {
        "fn": "statsforecast_intermittent",
        "sf_model": "CrostonSBA",
        "season_len": MONTHLY_SEASON_LENGTH,
    }
    SERIES_CONFIGS["SF-CrostonClassic"] = {
        "fn": "statsforecast_intermittent",
        "sf_model": "CrostonClassic",
        "season_len": MONTHLY_SEASON_LENGTH,
    }


def _build_global_models():
    out = {}
    if HAS_LIGHTGBM:
        out["LGBM-default"] = LGBMRegressor(
            n_estimators=500, max_depth=6, learning_rate=0.1, verbose=-1, random_state=42
        )
        out["LGBM-shallow"] = LGBMRegressor(
            n_estimators=300, max_depth=4, learning_rate=0.08, min_child_samples=15, verbose=-1, random_state=42
        )
    if HAS_XGB:
        out["XGB-MAE"] = XGBRegressor(
            n_estimators=400, max_depth=5, learning_rate=0.08, objective="reg:absoluteerror",
            verbosity=0, random_state=42, enable_categorical=True,
        )
        try:
            out["XGB-pseudohuber"] = XGBRegressor(
                n_estimators=400, max_depth=5, learning_rate=0.08,
                objective="reg:pseudohubererror", verbosity=0, random_state=42, enable_categorical=True,
            )
        except Exception:
            pass
    if HAS_CATBOOST:
        out["CatBoost-default"] = CatBoostRegressor(
            iterations=400, depth=6, learning_rate=0.08, loss_function="MAE",
            verbose=False, random_seed=42, allow_writing_files=False,
        )
    return out


In [ ]:
warning_rows: list = []
all_results: list = []
all_predictions: dict = {}
df_by_rid: dict = {}
mask_test_by_rid: dict = {}
cat_op_by_rid: dict = {}
cat_dt_by_rid: dict = {}
summary_parts: list[str] = []

RESIST_IDS = list(dict.fromkeys(RESIST_IDS))

for rid in RESIST_IDS:
    df = df_full[df_full["resist_id"] == rid].copy()
    df["date"] = pd.to_datetime(df["date"]).dt.normalize()
    mtest = (df["year"] == TEST_YEAR) & (df["month"].isin(TEST_MONTHS))
    mtrain = ~mtest
    df_by_rid[rid] = df
    mask_test_by_rid[rid] = mtest

    y_test = df.loc[mtest, "fact_value"].values
    n_test = int(mtest.sum())

    t_fp, s_fp, q_fp = (np.nan, np.nan, np.nan)
    if df["fact_predict"].notna().any():
        t_fp, s_fp, q_fp = calc_metrics(y_test, df.loc[mtest, "fact_predict"].values)

    train_rows = df.loc[mtrain].copy()
    X_tr = train_rows[FEATURE_COLS_NUM + CAT_COLS_ML].copy()
    X_tr["operation_id"] = X_tr["operation_id"].astype(str)
    X_tr["demand_type"] = X_tr["demand_type"].astype(str)
    y_tr = train_rows["fact_value"].values
    cat_op = sorted(X_tr["operation_id"].unique())
    cat_dt = sorted(X_tr["demand_type"].unique())
    cat_op_by_rid[rid] = cat_op
    cat_dt_by_rid[rid] = cat_dt

    test_dates = sorted(df.loc[mtest, "date"].unique())
    total_train_by_date = df.loc[mtrain].groupby("date")["fact_value"].sum().sort_index()
    y_tot_hist = total_train_by_date.values.astype(float)
    top_fc = forecast_total_top_ets(y_tot_hist, len(test_dates))

    results_rid: list = []
    preds_rid: dict = {}

    global_models = _build_global_models()
    for cfg_name, model in global_models.items():
        combo_key = f"{cfg_name} | {FEATURE_SET_NAME}"
        t0 = time.perf_counter()
        fill_xgb = cfg_name.startswith("XGB")
        cb_mode = cfg_name.startswith("CatBoost")
        try:
            if cb_mode and HAS_CATBOOST:
                import catboost as cb
                X_fit = pd.concat(
                    [train_rows[FEATURE_COLS_NUM].reset_index(drop=True), X_tr[CAT_COLS_ML].reset_index(drop=True)],
                    axis=1,
                )
                X_fit.columns = FEATURE_COLS_NUM + CAT_COLS_ML
                for c in FEATURE_COLS_NUM:
                    X_fit[c] = X_fit[c].fillna(0.0)
                pool_tr = cb.Pool(X_fit, label=y_tr, cat_features=CAT_COLS_ML)
                with warnings.catch_warnings(record=True) as w_fit:
                    warnings.simplefilter("always")
                    model.fit(pool_tr)
                append_warning_rows(
                    warning_rows, resist_id=rid, operation_id=TABULAR_OP_PLACEHOLDER,
                    model_kind="tabular", model_label=combo_key, phase="fit",
                    n_train_points=len(y_tr), caught=w_fit,
                )
                y_pred = recursive_global_predict(
                    model, df, mtrain, mtest, test_dates,
                    mean_volume_all_time, demand_type_map, first_active_train,
                    cat_op, cat_dt,
                    fill_nan_for_xgb=True, catboost_mode=True,
                )
            else:
                Xm = X_tr.copy()
                Xm["operation_id"] = pd.Categorical(Xm["operation_id"], categories=cat_op)
                Xm["demand_type"] = pd.Categorical(Xm["demand_type"], categories=cat_dt)
                if fill_xgb:
                    Xm[FEATURE_COLS_NUM] = Xm[FEATURE_COLS_NUM].fillna(0.0)
                X_fit = Xm[FEATURE_COLS_NUM + CAT_COLS_ML]
                with warnings.catch_warnings(record=True) as w_fit:
                    warnings.simplefilter("always")
                    model.fit(X_fit, y_tr)
                append_warning_rows(
                    warning_rows, resist_id=rid, operation_id=TABULAR_OP_PLACEHOLDER,
                    model_kind="tabular", model_label=combo_key, phase="fit",
                    n_train_points=len(y_tr), caught=w_fit,
                )
                y_pred = recursive_global_predict(
                    model, df, mtrain, mtest, test_dates,
                    mean_volume_all_time, demand_type_map, first_active_train,
                    cat_op, cat_dt,
                    fill_nan_for_xgb=fill_xgb, catboost_mode=False,
                )
        except Exception as ex:
            print(combo_key, ex)
            y_pred = np.zeros(n_test)
        elapsed = round(time.perf_counter() - t0, 2)
        t1, s1, q1 = calc_metrics(y_test, y_pred)
        results_rid.append({
            "resist_id": rid, "Конфигурация": cfg_name, "Набор признаков": FEATURE_SET_NAME,
            "Тип": "табличная", "Точность, %": t1, "Стабильность, %": s1, "Качество, %": q1,
            "Время, сек": elapsed, "Точн. fact_predict, %": t_fp, "Стаб. fact_predict, %": s_fp, "Кач. fact_predict, %": q_fp,
        })
        preds_rid[combo_key] = y_pred

    base_key = "LGBM-default | " + FEATURE_SET_NAME
    if base_key in preds_rid:
        t0 = time.perf_counter()
        y_base = preds_rid[base_key].copy()
        dtest = df.loc[mtest].reset_index(drop=True)
        y_rec = np.zeros_like(y_base, dtype=float)
        for mi, dt in enumerate(sorted(test_dates)):
            m = (dtest["date"] == dt).values
            bvec = y_base[m]
            y_rec[m] = top_down_reconcile_month(bvec, top_fc[mi])
        elapsed = round(time.perf_counter() - t0, 2)
        t1, s1, q1 = calc_metrics(y_test, y_rec)
        ck = "LGBM-default-TD-reconcile | " + FEATURE_SET_NAME
        results_rid.append({
            "resist_id": rid, "Конфигурация": "LGBM-default-TD-reconcile", "Набор признаков": FEATURE_SET_NAME,
            "Тип": "hierarchical", "Точность, %": t1, "Стабильность, %": s1, "Качество, %": q1,
            "Время, сек": elapsed, "Точн. fact_predict, %": t_fp, "Стаб. fact_predict, %": s_fp, "Кач. fact_predict, %": q_fp,
        })
        preds_rid[ck] = y_rec

    for config_name, cfg in SERIES_CONFIGS.items():
        t0 = time.perf_counter()
        if cfg["fn"] == "ets":
            y_pred = train_ets_per_series(
                df, mtrain, mtest, rid, config_name, warning_rows,
                trend=cfg["trend"], damped_trend=cfg["damped_trend"],
                seasonal=cfg.get("seasonal"), seasonal_periods=cfg.get("seasonal_periods"),
            )
        elif cfg["fn"] == "statsforecast_intermittent" and HAS_STATSFORECAST:
            y_pred = train_statsforecast_intermittent_per_series(
                df, mtrain, mtest, rid, config_name, warning_rows,
                sf_model_name=cfg["sf_model"], season_len=cfg["season_len"],
            )
        else:
            continue
        elapsed = round(time.perf_counter() - t0, 2)
        t1, s1, q1 = calc_metrics(y_test, y_pred)
        results_rid.append({
            "resist_id": rid, "Конфигурация": config_name, "Набор признаков": "—",
            "Тип": "per-series", "Точность, %": t1, "Стабильность, %": s1, "Качество, %": q1,
            "Время, сек": elapsed, "Точн. fact_predict, %": t_fp, "Стаб. fact_predict, %": s_fp, "Кач. fact_predict, %": q_fp,
        })
        preds_rid[config_name] = y_pred

    all_results.extend(results_rid)
    all_predictions[rid] = preds_rid
    q_best = max(r["Качество, %"] for r in results_rid)
    summary_parts.append(f"{rid} forecast-best:{q_best:.0f}%")
    print(", ".join(summary_parts))

results = all_results


#### Опционально: ZIP по **агрегированному** месячному total (train-only демо)

`statsmodels` **ZeroInflatedPoisson** на ряду сумм по месяцу; если не сходится — строка в предупреждениях.


In [ ]:
try:
    from statsmodels.discrete.count_model import ZeroInflatedPoisson

    for rid in RESIST_IDS:
        df = df_by_rid[rid]
        mtrain = ~((df["year"] == TEST_YEAR) & (df["month"].isin(TEST_MONTHS)))
        tot = df.loc[mtrain].groupby("date")["fact_value"].sum().astype(float)
        y_agg = tot.values
        X_agg = np.ones((len(y_agg), 1))
        with warnings.catch_warnings(record=True) as wz:
            warnings.simplefilter("always")
            z = ZeroInflatedPoisson(y_agg.clip(min=0), X_agg)
            res = z.fit(disp=False, maxiter=100)
        append_warning_rows(
            warning_rows, resist_id=rid, operation_id=TABULAR_OP_PLACEHOLDER,
            model_kind="tabular", model_label="ZIP-total-train-only", phase="fit",
            n_train_points=len(y_agg), caught=wz,
        )
        print("ZIP total fit OK:", rid, res.aic if hasattr(res, "aic") else "")
except Exception as e:
    print("ZIP-total пропущен:", e)


#### Сводная таблица результатов

In [ ]:
df_results = pd.DataFrame(results)
df_results["Комбинация"] = df_results.apply(
    lambda r: f"{r['Конфигурация']} | {r['Набор признаков']}" if r["Набор признаков"] != "—" else r["Конфигурация"],
    axis=1,
)
df_results_full = df_results.sort_values(
    ["resist_id", "Качество, %"], ascending=[True, False]
).reset_index(drop=True)
df_results = (
    df_results_full.groupby("resist_id", group_keys=False)
    .head(SUMMARY_TOP_PER_RID)
    .reset_index(drop=True)
)
print(f"Строк в полной сводке: {len(df_results_full)}; в таблице — топ-{SUMMARY_TOP_PER_RID} на resist_id")
df_results


In [ ]:
TOP_N = SUMMARY_TOP_PER_RID
for rid in RESIST_IDS:
    sub = df_results_full[df_results_full["resist_id"] == rid].head(TOP_N)
    if sub.empty:
        continue
    fig, axes = plt.subplots(1, 3, figsize=(18, max(6, TOP_N * 0.45)))
    metrics_to_plot = ["Точность, %", "Стабильность, %", "Качество, %"]
    for ax, metric in zip(axes, metrics_to_plot):
        df_plot = sub.sort_values(metric, ascending=True)
        colors = ["#2196F3" if t == "табличная" else ("#9C27B0" if t == "hierarchical" else "#FF9800") for t in df_plot["Тип"]]
        ax.barh(df_plot["Комбинация"], df_plot[metric], color=colors)
        ax.set_xlabel(metric)
        ax.set_title(metric)
    plt.suptitle(f"TOP-{TOP_N} (resist_id={rid})", fontsize=14)
    plt.tight_layout()
    plt.show()


#### Лучшая конфигурация: детали по операциям

In [ ]:
per_rid_best = {}

for rid in RESIST_IDS:
    df_r = df_results_full[df_results_full["resist_id"] == rid].sort_values(
        "Качество, %", ascending=False
    )
    if df_r.empty:
        continue
    best_row = df_r.iloc[0]
    best_combo = best_row["Комбинация"]
    best_config = best_row["Конфигурация"]
    best_fs = best_row["Набор признаков"]
    best_type = best_row["Тип"]
    print(f"\n{'='*60}\nresist_id={rid}\nЛучшая комбинация: {best_combo} ({best_type})")

    best_pred = all_predictions[rid][best_combo]
    df = df_by_rid[rid]
    mask_test = mask_test_by_rid[rid]
    df_test = df.loc[mask_test].copy()
    df_test["predict_new"] = best_pred

    rows_op = []
    for (r_id, op_id), grp in df_test.groupby(["resist_id", "operation_id"]):
        t2, s2, q2 = calc_metrics(grp["fact_value"].values, grp["predict_new"].values)
        row = {
            "resist_id": r_id,
            "operation_id": op_id,
            f"Точность ({best_combo})": t2,
            f"Стабильность ({best_combo})": s2,
            f"Качество ({best_combo})": q2,
        }
        if grp["fact_predict"].notna().any():
            t1, s1, q1 = calc_metrics(grp["fact_value"].values, grp["fact_predict"].values)
            row["Точность (fact_predict)"] = t1
            row["Стабильность (fact_predict)"] = s1
            row["Качество (fact_predict)"] = q1
        rows_op.append(row)

    df_metrics_op = pd.DataFrame(rows_op)
    df_detail = df_test[["resist_id", "operation_id", "year", "month",
                         "fact_value", "fact_predict", "predict_new"]].copy()
    df_detail = df_detail.sort_values(["resist_id", "operation_id", "year", "month"]).reset_index(drop=True)
    df_detail = df_detail.rename(columns={
        "fact_value": "fact_value (факт)",
        "fact_predict": "fact_predict (текущий)",
        "predict_new": f"{best_combo} (новый)",
    })
    per_rid_best[rid] = {
        "best_combo": best_combo,
        "df_metrics_op": df_metrics_op,
        "df_detail": df_detail,
    }
    print(f"Операций в тесте: {len(df_metrics_op)}")

if RESIST_IDS:
    r0 = RESIST_IDS[0]
    print("\n(пример df_metrics_op.head())")
    print(per_rid_best[r0]["df_metrics_op"].head(10))


#### Экспорт в Excel

In [ ]:
from openpyxl import load_workbook

out_path = OUT_XLSX
out_path.parent.mkdir(parents=True, exist_ok=True)

WARN_COLS = [
    "resist_id", "operation_id", "model_kind", "model_label", "phase",
    "warning_category", "message", "interpretation", "n_train_points", "statsmodels_module",
]

sheet_triples = []
for rid in RESIST_IDS:
    df_itog = df_results_full[df_results_full["resist_id"] == rid].head(EXCEL_ITOG_TOP)
    if rid not in per_rid_best:
        continue
    b = per_rid_best[rid]
    df_warn = pd.DataFrame([r for r in warning_rows if r["resist_id"] == rid])
    if df_warn.empty:
        df_warn = pd.DataFrame(columns=WARN_COLS)
    else:
        df_warn = df_warn.reindex(columns=WARN_COLS)
    sheet_triples.append((rid, df_itog, b["df_detail"], b["df_metrics_op"], df_warn))

names_to_remove = []
for rid, _, _, _, _ in sheet_triples:
    names_to_remove.extend(
        [f"{rid} Итог", f"{rid} Данные теста", f"{rid} По операциям", f"{rid} Предупреждения"]
    )

if out_path.exists():
    wb = load_workbook(out_path)
    for sn in names_to_remove:
        if sn in wb.sheetnames:
            wb.remove(wb[sn])
    if len(wb.sheetnames) == 0:
        wb.create_sheet("_tmp")
    wb.save(out_path)
    wb.close()
    writer_mode = "a"
else:
    writer_mode = "w"

with pd.ExcelWriter(out_path, engine="openpyxl", mode=writer_mode) as w:
    for rid, df_itog, df_detail, df_metrics_op, df_warn in sheet_triples:
        df_itog.to_excel(w, sheet_name=f"{rid} Итог", index=False)
        df_detail.to_excel(w, sheet_name=f"{rid} Данные теста", index=False)
        df_metrics_op.to_excel(w, sheet_name=f"{rid} По операциям", index=False)
        df_warn.to_excel(w, sheet_name=f"{rid} Предупреждения", index=False)

wb = load_workbook(out_path)
if "_tmp" in wb.sheetnames and len(wb.sheetnames) > 1:
    wb.remove(wb["_tmp"])
    wb.save(out_path)
wb.close()

print(f"Excel: {out_path.resolve()}")


#### Проверка на утечку (data leakage)

1. Статика (`demand_type`, `mean_volume_all_time`, `first_active` для age) считается только по **train** (`train_static_per_operation(df_train_only)`).
2. Рекурсивный прогноз не подставляет факт теста в `extended` — только прогнозы прошлых шагов теста.
3. Ниже — мини-тест `global_volume_lag_1` и сравнение с **наивной утечкой** (leaked): одноразовый `predict` по фичам из `df_full` на тесте (фактические лаги на тесте).


In [ ]:
# Мини-тест: две операции, два месяца теста — руками проверяем global lag
_mini = pd.DataFrame({
    "resist_id": [1, 1, 1, 1],
    "operation_id": [10, 20, 10, 20],
    "date": pd.to_datetime(["2025-01-01", "2025-01-01", "2025-02-01", "2025-02-01"]),
    "year": [2025, 2025, 2025, 2025],
    "month": [1, 1, 2, 2],
    "fact_value": [1.0, 2.0, 3.0, 4.0],
    "fact_predict": [np.nan] * 4,
})
_mv = _mini.groupby("operation_id")["fact_value"].mean()
_dem = {10: "Smooth", 20: "Smooth"}
_fa = {10: pd.Timestamp("2025-01-01"), 20: pd.Timestamp("2025-01-01")}
_f = add_panel_features(_mini, _mv, _dem, _fa)
assert abs(_f.loc[_f["date"] == pd.Timestamp("2025-02-01"), "global_volume_lag_1"].iloc[0] - 3.0) < 1e-6
print("global_volume_lag_1 mini-test OK")

# Leaked upper bound: одношаговый predict по фичам df_full на тесте (факт в лагах теста)
if HAS_LIGHTGBM and RESIST_IDS:
    try:
        rid = RESIST_IDS[0]
        df = df_by_rid[rid]
        mtest = mask_test_by_rid[rid]
        mtrain = ~mtest
        tr = df.loc[mtrain]
        cop = cat_op_by_rid.get(rid) or sorted(tr["operation_id"].astype(str).unique())
        cdt = cat_dt_by_rid.get(rid) or sorted(tr["demand_type"].astype(str).unique())
        X_leak = df.loc[mtest][FEATURE_COLS_NUM + CAT_COLS_ML].copy()
        X_leak["operation_id"] = pd.Categorical(X_leak["operation_id"].astype(str), categories=cop)
        X_leak["demand_type"] = pd.Categorical(X_leak["demand_type"].astype(str), categories=cdt)
        y_tr = tr["fact_value"].values
        Xm = tr[FEATURE_COLS_NUM + CAT_COLS_ML].copy()
        Xm["operation_id"] = pd.Categorical(Xm["operation_id"].astype(str), categories=cop)
        Xm["demand_type"] = pd.Categorical(Xm["demand_type"].astype(str), categories=cdt)
        if len(Xm) < 2:
            raise ValueError("мало train для диагностики")
        m = LGBMRegressor(n_estimators=200, max_depth=5, learning_rate=0.1, verbose=-1, random_state=42)
        m.fit(Xm, y_tr)
        pred_leak = np.maximum(m.predict(X_leak), 0)
        t_l, s_l, q_l = calc_metrics(df.loc[mtest, "fact_value"].values, pred_leak)
        print(f"Диагностика (НЕ для отчёта): leaked one-shot LGBM Качество={q_l}% (обычно завышено vs рекурсии)")
    except Exception as _e:
        print("Диагностика leaked baseline пропущена:", _e)
